In [ ]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.utils import resample

In [ ]:
df.to_csv('final_cleaned_twitter_dataset.csv', index=False, encoding='utf-8-sig')

In [ ]:
df['tweet'] = df['tweet'].str.replace(r'•\.', '', regex=True)
df['tweet'] = df['tweet'].str.replace(r'≧∇≦', '', regex=True)
df['tweet'] = df['tweet'].str.replace(r'•', '', regex=True)

df = df[df['tweet'].str.count(r'[a-zA-Z]') >= 3]

df['tweet'] = df['tweet'].str.replace(r'\s+', ' ', regex=True).str.strip()

print(f"Junk rows removed! Remaining rows: {len(df)}")

In [ ]:
val_df['tweet'] = val_df['tweet'].str.replace(r'•\.', '', regex=True)
val_df['tweet'] = val_df['tweet'].str.replace(r'≧∇≦', '', regex=True)


val_df = val_df[val_df['tweet'].str.count(r'[a-zA-Z]') >= 3]

val_df['tweet'] = val_df['tweet'].str.replace(r'\s+', ' ', regex=True).str.strip()

print(f"Junk rows removed! Remaining rows: {len(val_df)}")

In [ ]:
df = pd.read_csv('twitter_training.csv')

df.columns = ['tweet_id', 'entity', 'sentiment', 'tweet']
df.head()

### 3. Basic Data Inspection

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

In [ ]:
df["tweet"] = df["tweet"].str.strip()

### 4. Handle Missing Values


In [ ]:
# Drop rows where tweet is missing
df = df.dropna(subset=['tweet'])

# Reset index
df = df.reset_index(drop=True)

### 5. Normalize Sentiment Labels

In [ ]:
df['sentiment'] = df['sentiment'].str.lower().str.strip()
print(df['sentiment'].value_counts())

### 6. Remove Duplicates

In [ ]:
# Remove exact duplicates
df = df.drop_duplicates(subset=['tweet'])

# Reset index
df = df.reset_index(drop=True)

### 7. Remove Very Short Tweets



In [ ]:
df['word_count'] = df['tweet'].apply(lambda x: len(str(x).split()))
df = df[df['word_count'] >= 3]
df = df.drop(columns=['word_count'])

### 8. Text Cleaning Function

In [ ]:
!pip install contractions num2words ftfy emoji

In [ ]:
import string
import contractions
from num2words import num2words
from ftfy import fix_text
import emoji

In [ ]:
# Fix encoding issues like â€™ → '
def fix_encoding(text):

    text = fix_text(text)

    text = re.sub(r'â€™\s*', "'", text)

    return text

In [ ]:
def convert_emojis(text):
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")
    return text

In [ ]:
# Handle informal contractions like im, dont, etc.
def normalize_informal(text):
    text = re.sub(r'\bim\b', 'i am', text)
    text = re.sub(r'\bive\b', 'i have', text)
    text = re.sub(r'\bill\b', 'i will', text)
    text = re.sub(r'\bdont\b', 'do not', text)
    text = re.sub(r'\bdoesnt\b', 'does not', text)
    text = re.sub(r'\bcant\b', 'cannot', text)
    text = re.sub(r'\bwont\b', 'will not', text)
    return text

In [ ]:
# Convert numbers to words
def convert_numbers(text):
    def replace_number(match):
        return num2words(int(match.group()))
    return re.sub(r'\d+', replace_number, text)

In [ ]:
# Remove URLs (normal + short + broken spaced ones)
def remove_urls(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\b\w+\.\w+/\S*', '', text)
    text = re.sub(r'\b\w+\s*\.\s*\w+\s*/\s*\S*', '', text)
    return text


In [ ]:
def normalize_repetitions(text):
    # fix unicode leftovers (safety net)
    text = text.replace('â€¼', '!')
    text = text.replace('‼', '!')

    # remove mixed !. → !
    text = re.sub(r'!\s*\.', '!', text)

    # normalize spaced or repeated dots
    text = re.sub(r'(\.\s*){2,}', '. ', text)

    # normalize spaced or repeated !
    text = re.sub(r'(!\s*){2,}', '!', text)

    # normalize character elongation (yessssss → yess)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    text = re.sub(r'^[.!]+\s*', '', text)

    return text

In [ ]:
# Remove punctuation but keep . and !
def remove_punctuation(text):
    punct = string.punctuation.replace('.', '').replace('!', '')
    return text.translate(str.maketrans('', '', punct))


# Clean extra spaces
def clean_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
def remove_unk(text):
    return " ".join([w for w in text.split() if w != "<unk>"])

In [ ]:
def clean_text(text):
    text = text.lower()

    text = fix_encoding(text)

    text = convert_emojis(text)

    text = remove_unk(text)

    text = re.sub(r'[@#]\s*\w+', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    text = normalize_informal(text)

    text = contractions.fix(text)

    text = convert_numbers(text)

    text = remove_urls(text)


    text = normalize_repetitions(text)

    text = remove_punctuation(text)

    text = re.sub(r'(\.\s*){2,}', '. ', text)

    text = clean_spaces(text)

    return text

In [ ]:
df["tweet"] = df["tweet"].apply(lambda x: clean_text(x) if isinstance(x, str) else x)

In [ ]:
df['tweet'].head(10)

In [ ]:
df["tweet"] = df["tweet"].apply(lambda x: clean_text(x) if isinstance(x, str) else "")

In [ ]:
df = df[df["tweet"].apply(lambda x: bool(re.search(r'[a-z]', x)))].reset_index(drop=True)

### Removing Duplicates After Cleaning

In [ ]:
duplicate_count = df["tweet"].duplicated().sum()
print("Number of duplicate tweets:", duplicate_count)

In [ ]:
duplicates = df[df["tweet"].duplicated(keep=False)]
duplicates.head()

In [ ]:
df = df.drop_duplicates(subset=["tweet"]).reset_index(drop=True)

### Language Detection

In [ ]:
!pip install fasttext

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin

In [ ]:
!pip install numpy==1.26.4

In [ ]:
import fasttext

model = fasttext.load_model("lid.176.bin")

df["language"] = df["tweet"].apply(
    lambda x: model.predict(x)[0][0].replace("__label__", "")
)

In [ ]:
df["language"].value_counts()

In [ ]:
rare_langs = df["language"].value_counts()[df["language"].value_counts() < 10].index

for lang in rare_langs:
    print(f"\n--- {lang} ---")
    print(df[df["language"] == lang]["tweet"].head(3).to_string(index=False))

In [ ]:
df = df[df["language"] == "en"].reset_index(drop=True)

In [ ]:
df.shape

In [ ]:
df.drop(columns = ['language'],inplace = True)

### Checking If there are differenet context tweets for one tweet id and choosing the meaningful tweets

In [ ]:

def calculate_max_word_difference(group):
    # Get all texts for this Tweet ID, convert to lowercase
    texts = group['tweet'].str.lower().tolist()

    # If there's only one variation, there's no difference
    if len(texts) <= 1:
        return 0

    # Convert each text into a set of unique words
    word_sets = [set(text.split()) for text in texts]

    # Find the longest text (in terms of unique words) to act as our "baseline"
    baseline_set = max(word_sets, key=len)

    max_diff = 0
    for ws in word_sets:
        # Calculate how many words in this variation are NOT in the baseline text
        # (Using standard set difference: elements in ws but not in baseline_set)
        diff_count = len(ws - baseline_set)

        if diff_count > max_diff:
            max_diff = diff_count

    return max_diff


word_differences = df.groupby('tweet_id').apply(calculate_max_word_difference)

# 4. Set your threshold and filter
THRESHOLD = 4 # Flag if a variation introduces more than 3 completely new words
flagged_ids = word_differences[word_differences > THRESHOLD].index

print(f"Found {len(flagged_ids)} Tweet IDs exceeding the word difference threshold of {THRESHOLD}.")

In [ ]:
manual_review_df = df[df['tweet_id'].isin(flagged_ids)]

# Grab 3 random Tweet IDs from the flagged list
sample_ids = manual_review_df['tweet_id'].drop_duplicates().sample(n=3, random_state=42)

# Filter down to just those 3 IDs
sample_df = manual_review_df[manual_review_df['tweet_id'].isin(sample_ids)]

# Print them out grouped by ID
for t_id, group in sample_df.groupby('tweet_id'):
    print(f"\n=== Tweet ID: {t_id} ===")
    for index, row in group.iterrows():
        print(f"- {row['tweet']}")

In [ ]:
# Grab 3 random Tweet IDs from the flagged list
sample_ids = manual_review_df['tweet_id'].drop_duplicates().sample(n=3, random_state=42)

# Filter down to just those 3 IDs
sample_df = manual_review_df[manual_review_df['tweet_id'].isin(sample_ids)]

for t_id, group in sample_df.groupby('tweet_id'):
    print(f"\n=== Tweet ID: {t_id} ===")

    # Run the exact logic we used in the function to find the "best" index
    lengths = group['tweet'].str.len()
    median_length = lengths.median()
    closest_idx = (lengths - median_length).abs().idxmin()

    # Print all rows, marking the one the script chose
    for index, row in group.iterrows():
        text = row['tweet']
        char_len = len(text)

        if index == closest_idx:
            print(f"--> SELECTED [Len: {char_len}]: {text}")
        else:
            print(f"    IGNORED  [Len: {char_len}]: {text}")

#### using embedding to check the filtered ids

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# 1. BATCH ENCODING: Extract unique tweets and encode them all at once.
# batch_size=128 maxes out CPU efficiency. show_progress_bar gives you a nice loading bar.
unique_tweets = df['tweet'].unique().tolist()
print(f"Encoding {len(unique_tweets)} unique text strings...")
embeddings = model.encode(unique_tweets, batch_size=128, show_progress_bar=True)

# Create a fast lookup dictionary: { "tweet_text": [vector_array] }
embedding_dict = dict(zip(unique_tweets, embeddings))

# 2. SCORING FUNCTION: Now it just does math instantly using the pre-computed vectors.
def get_minimum_semantic_similarity(group):
    texts = group['tweet'].tolist()
    if len(texts) <= 1:
        return 1.0

    # Instantly grab the pre-calculated vectors from our dictionary
    group_embeddings = [embedding_dict[text] for text in texts]

    # Calculate similarity matrix and return the lowest score
    sim_matrix = cosine_similarity(group_embeddings)
    return sim_matrix.min()

# 3. EXECUTE: Apply to the dataframe
print("Calculating similarity scores...")
similarities = df.groupby('tweet_id').apply(get_minimum_semantic_similarity)

# 4. FILTER: Isolate the heavily mismatched IDs
SEMANTIC_THRESHOLD = 0.75
semantic_flagged_ids = similarities[similarities < SEMANTIC_THRESHOLD].index

print(f"Found {len(semantic_flagged_ids)} IDs with semantically mismatched text.")

In [ ]:
# Grab 3 random Tweet IDs from the semantic flagged list
semantic_sample_ids = np.random.choice(semantic_flagged_ids, 6, replace=False)

# Filter down to just those 3 IDs
sample_df = df[df['tweet_id'].isin(semantic_sample_ids)]

for t_id, group in sample_df.groupby('tweet_id'):
    print(f"\n=== Tweet ID: {t_id} ===")
    for index, row in group.iterrows():
        print(f"- {row['tweet']}")

In [ ]:
semantic_sample_ids = np.random.choice(semantic_flagged_ids, 6, replace=False)

# Filter down to just those 3 IDs
sample_df = df[df['tweet_id'].isin(semantic_sample_ids)]

for t_id, group in sample_df.groupby('tweet_id'):
    print(f"\n=== Tweet ID: {t_id} ===")

    texts = group['tweet'].tolist()

    # If there happens to be only one row, it wins by default
    if len(texts) <= 1:
         print(f"--> SELECTED (Only option): {texts[0]}")
         continue

    # Grab the pre-calculated vectors from our dictionary
    group_embeddings = [embedding_dict[text] for text in texts]

    # Calculate consensus math
    sim_matrix = cosine_similarity(group_embeddings)

    # The sum of similarities across the row gives us the total consensus score
    consensus_scores = sim_matrix.sum(axis=1)
    best_idx = consensus_scores.argmax()

    # Print results, marking the chosen row
    for i, text in enumerate(texts):
        score = consensus_scores[i]
        if i == best_idx:
            print(f"--> SELECTED [Consensus Score: {score:.2f}]: {text}")
        else:
            print(f"    IGNORED  [Consensus Score: {score:.2f}]: {text}")

In [ ]:
semantic_sample_ids = np.random.choice(semantic_flagged_ids, 6, replace=False)

sample_df = df[df['tweet_id'].isin(semantic_sample_ids)]

for t_id, group in sample_df.groupby('tweet_id'):
    print(f"\n=== Tweet ID: {t_id} ===")

    texts = group['tweet'].tolist()
    if len(texts) <= 1:
         print(f"--> SELECTED (Only option): {texts[0]}")
         continue

    # 1. Semantic Math
    group_embeddings = [embedding_dict[text] for text in texts]
    sim_matrix = cosine_similarity(group_embeddings)
    consensus_scores = sim_matrix.sum(axis=1)

    # 2. Find valid candidates (within 2% of max score)
    max_score = consensus_scores.max()
    threshold = max_score * 0.98
    valid_indices = np.where(consensus_scores >= threshold)[0]

    # 3. Length Tie-Breaker
    group_median_len = group['tweet'].str.len().median()

    best_idx = valid_indices[0]
    smallest_length_diff = float('inf')

    for idx in valid_indices:
        len_diff = abs(len(texts[idx]) - group_median_len)
        if len_diff < smallest_length_diff:
            smallest_length_diff = len_diff
            best_idx = idx

    # Print the results
    for i, text in enumerate(texts):
        score = consensus_scores[i]
        char_len = len(text)

        if i == best_idx:
            print(f"--> SELECTED   [Score: {score:.2f} | Len: {char_len}]: {text}")
        elif i in valid_indices:
            print(f"    TIE-BROKEN [Score: {score:.2f} | Len: {char_len}]: {text}")
        else:
            print(f"    IGNORED    [Score: {score:.2f} | Len: {char_len}]: {text}")

    print(f"    (Group Median Length was: {group_median_len})")

#### choosing the best variation of tweet

In [ ]:
def get_best_variation(group):
    texts = group['tweet'].tolist()

    # If there is only one variation, it automatically wins
    if len(texts) <= 1:
        return group.iloc[0]

    # 1. Semantic Consensus (Fast lookup from dictionary)
    group_embeddings = [embedding_dict[text] for text in texts]
    sim_matrix = cosine_similarity(group_embeddings)
    consensus_scores = sim_matrix.sum(axis=1)

    # 2. Find valid candidates (within 2% of the highest consensus score)
    max_score = consensus_scores.max()
    threshold = max_score * 0.98
    valid_indices = np.where(consensus_scores >= threshold)[0]

    # 3. Median Length Tie-Breaker
    group_median_len = group['tweet'].str.len().median()

    best_idx = valid_indices[0]
    smallest_length_diff = float('inf')

    for idx in valid_indices:
        len_diff = abs(len(texts[idx]) - group_median_len)
        if len_diff < smallest_length_diff:
            smallest_length_diff = len_diff
            best_idx = idx

    return group.iloc[best_idx]

# Apply to the ENTIRE original dataframe
print(f"Cleaning all {len(df)} rows...")
final_clean_df = df.groupby('tweet_id', group_keys=False).apply(get_best_variation)

# Save the pristine dataset for fine-tuning
final_clean_df.to_csv('final_cleaned_twitter_dataset.csv', index=False)

print(f"Done! Extracted {len(final_clean_df)} perfect rows for Llama-3.")

### Checking if the small sentences having less then 5 words are useful

In [ ]:
!pip install vaderSentiment


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

analyzer = SentimentIntensityAnalyzer()

def filter_meaningless_shorts(text):
    # Standardize to string and split into words
    words = str(text).split()

    # Logic 1: If it's 5 words or longer, we keep it for context.
    if len(words) >= 8:
        return True

    # Logic 2: If it's short (< 5 words), check for sentiment.
    # The 'compound' score ranges from -1 (negative) to 1 (positive).
    # 0 means the model found no sentiment-charged words.
    scores = analyzer.polarity_scores(text)

    # We keep it only if there is a non-zero sentiment detected.
    # You can adjust '0.05' to be more or less strict.
    return abs(scores['compound']) > 0.05

# Apply the filter to your already-cleaned dataframe
print(f"Rows before short-text filtering: {len(final_clean_df)}")

final_clean_df = final_clean_df[final_clean_df['tweet'].apply(filter_meaningless_shorts)]

print(f"Rows after short-text filtering: {len(final_clean_df)}")

### Merging Irrelevant and Neutral to others

In [ ]:
final_clean_df['sentiment'] = final_clean_df['sentiment'].replace({
    'Irrelevant': 'Others',
    'Neutral': 'Others'
})

In [ ]:
print(final_clean_df['sentiment'].value_counts())

### Data Spliting

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    final_clean_df,
    test_size=0.2,
    random_state=42,
    stratify=final_clean_df["sentiment"]
)

In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["sentiment"]
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train_df["sentiment"] = le.fit_transform(train_df["sentiment"])
val_df["sentiment"] = le.transform(val_df["sentiment"])
test_df["sentiment"] = le.transform(test_df["sentiment"])

In [ ]:
train_df.to_csv("train_scleaned.csv", index=False)

In [ ]:
val_df.to_csv("val_scleaned.csv", index=False)

In [ ]:
test_df.to_csv("val_scleaned.csv", index=False)